# Phase 1 — Hoang Intent-Aware Routing Research

Notebook mô tả **toàn bộ luồng Phase 1** trước retrieval, đồng bộ với `aviation_rag/phase1_hoang_intent_routing.py` và Streamlit.

**Mục tiêu:** nhận **user query ngắn** → phân loại **intent** → sinh **retrieval plan** cho Phase 2.

**Nội dung chính**

1. Train **TF-IDF + Logistic Regression** trên **query-only corpus** (không train trên ASRS narrative).
2. Đánh giá trên **gold-set 8 câu** (human label, held-out khỏi training).
3. So sánh **ML vs heuristic** và runtime mode **`auto`** (ML + fallback).
4. Sinh **retrieval plan** (strategy, fallback, filters) theo intent.
5. Demo BM25 vs semantic trên query procedure.
6. Export contract `phase1_hoang_intent_routing_output.jsonl` cho Phase 2.


In [1]:
from pathlib import Path
import json
import sys
from dataclasses import replace
from statistics import mean

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        if hasattr(obj, "to_string"):
            print(obj.to_string())
        else:
            print(obj)

ROOT = Path.cwd().resolve()
if not (ROOT / 'aviation_rag').exists() and (ROOT.parent / 'aviation_rag').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from aviation_rag.config import Settings, ensure_artifact_dirs
from aviation_rag.io_utils import write_jsonl, read_jsonl
from aviation_rag.phase1_hoang_intent_routing import Phase1HoangIntentRouting, heuristic_intent
from aviation_rag.phase1_intent_training import (
    evaluate_gold_labels,
    evaluate_heuristic_gold_labels,
    training_report_summary,
)
from aviation_rag.phase2_san_faiss_retrieval import Phase2SanFaissRetrieval

base_settings = Settings()
settings = replace(base_settings, input_intent_mode='auto', phase1_retrain=True)
ensure_artifact_dirs(settings)

phase1_preview = Phase1HoangIntentRouting(settings)
report = phase1_preview.intent_training_report or {}
validation = report.get('validation_metrics', {})
gold = report.get('gold_metrics', {})
heuristic_gold = report.get('heuristic_gold_metrics', {})
corpus = report.get('training_corpus', {})

print('Project root:', settings.project_root)
print('Training queries:', settings.phase1_training_queries_path)
print('Gold labels:', settings.phase1_gold_labels_path)
print('Phase 1 artifact:', settings.phase1_output_path)
print('Saved model dir:', settings.phase1_model_dir)
print('---')
print('Intent runtime mode:', settings.input_intent_mode)
print('Training corpus type:', corpus.get('type'))
print('Training rows:', phase1_preview.intent_model.training_rows)
print('Training label counts:', phase1_preview.intent_model.label_counts)
print('Gold holdout skipped rows:', corpus.get('gold_holdout_skipped_rows'))
print('---')
print('Validation accuracy:', validation.get('validation_accuracy'))
print('Validation macro F1:', validation.get('validation_macro_f1'))
print('Gold accuracy (ML):', gold.get('gold_accuracy'), f"({gold.get('gold_rows')} queries)")
print('Gold accuracy (heuristic):', heuristic_gold.get('gold_accuracy'))
print('Summary:', training_report_summary(phase1_preview.intent_model))


Project root: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project
Training queries: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\data\phase1_intent_training_queries.jsonl
Gold labels: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\data\phase1_intent_gold_labels.jsonl
Phase 1 artifact: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\artifacts\phase1_hoang_intent_routing_output.jsonl
Saved model dir: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\artifacts\phase1_intent_model
---
Intent runtime mode: auto
Training corpus type: query_only
Training rows: 231
Training label counts: {'Incident_Report': 50, 'Technical_Procedure': 85, 'Metadata_Query': 49, 'Factoid': 47}
Gold holdout skipped rows: 6
---
Validation accuracy: 0.9787
Validation macro F1: 0.9749
Gold accuracy (ML): 1.0 (8 queries)
Gold accuracy (heuristic): 1.0
Summary: {'

## Train / Validation / Gold-set — Định nghĩa & giải thích

### 1. Bài toán Phase 1 học gì?

| Khái niệm | Định nghĩa |
|-----------|------------|
| **Input lúc infer** | Câu hỏi ngắn của user (vd. *engine failure after takeoff*) |
| **Output** | Một trong 4 **intent** + `retrieval_plan` cho Phase 2 |
| **Intent** | Loại thông tin user muốn: narrative sự cố, procedure, metadata, hay factoid |

**Lưu ý quan trọng:** Phase 1 **không** phân loại “loại báo cáo ASRS dài”. Model phải học trên **câu query**, không phải narrative CSV.

---

### 2. Bốn intent (taxonomy)

| Intent | Ý nghĩa | Ví dụ query | Strategy mặc định |
|--------|---------|-------------|-------------------|
| `Incident_Report` | Tìm báo cáo sự cố / narrative tương tự | engine failure after takeoff | `semantic` |
| `Technical_Procedure` | Checklist, QRH, troubleshooting, MEL | engine warning checklist | `bm25` |
| `Metadata_Query` | Thời tiết, runway, icing, airport condition | crosswind turbulence final approach | `metadata_first` |
| `Factoid` | Định nghĩa / câu hỏi ngắn | what is MEL in aviation | `semantic` |

---

### 3. Training corpus — `query_only`

**Nguồn train** (gộp lại, loại trùng theo normalized text):

1. **Seed queries** — ~29 câu hard-code trong `phase1_hoang_intent_routing.py`
2. **`data/phase1_intent_training_queries.jsonl`** — câu bổ sung do team viết
3. **Augmentation** — biến thể prefix/suffix (*find asrs reports about …*)

**Không dùng:** cột `report_summary` / narrative ASRS làm text train (pseudo-label cũ gây lệch task).

**Preprocessing trước TF-IDF:** `normalize_text` → mở rộng jargon hàng không → **Snowball stemming** (EN).

---

### 4. Gold-set vs Validation-set (khác nhau)

| Tập | File | Số câu | Vai trò |
|-----|------|--------|---------|
| **Gold-set** | `data/phase1_intent_gold_labels.jsonl` | 8 | Human label; **held-out** — không vào training (exact match) |
| **Validation split** | Tách từ training corpus | ~20% (~47/231) | Đo ML trên query train cùng phân phối, stratified 80/20 |

- **Validation accuracy** ≈ model fit trên query train — **không thay** gold-set.
- **Gold accuracy** ≈ chất lượng thực trên 8 câu demo — **metric chính** để báo cáo Phase 1.

---

### 5. Runtime routing — ba mode (`INPUT_INTENT_MODE`)

| Mode | Hành vi |
|------|---------|
| **`auto`** (mặc định) | Chạy ML; nếu ML và heuristic **khác nhau** và `ml_confidence < 0.55` → dùng **heuristic** |
| **`ml`** | Luôn dùng TF-IDF + Logistic Regression |
| **`heuristic`** | Chỉ rule `heuristic_intent()` — không cần model fit tốt |

`intent_source` trong artifact: `"ml"` hoặc `"heuristic"` (mode thực tế đã chọn).

---

### 6. Artifact đã lưu

Thư mục `artifacts/phase1_intent_model/`:

- `preprocessing_pipeline.joblib` — cấu hình stem/normalize
- `tfidf_vectorizer.joblib` — vocabulary + IDF
- `logistic_classifier.joblib` — OneVsRest + LogisticRegression
- `training_report.json` — metrics, label counts, gold eval rows


In [2]:
gold_metrics = evaluate_gold_labels(
    phase1_preview.intent_model,
    settings.phase1_gold_labels_path,
    use_stemming=settings.phase1_use_stemming,
)
heuristic_gold_metrics = evaluate_heuristic_gold_labels(settings.phase1_gold_labels_path)
validation = (phase1_preview.intent_training_report or {}).get('validation_metrics', {})
corpus = (phase1_preview.intent_training_report or {}).get('training_corpus', {})

summary_rows = [
    {'metric': 'training_corpus_type', 'value': corpus.get('type')},
    {'metric': 'training_rows_total', 'value': phase1_preview.intent_model.training_rows},
    {'metric': 'train_rows_split', 'value': validation.get('train_rows')},
    {'metric': 'validation_rows_split', 'value': validation.get('validation_rows')},
    {'metric': 'validation_accuracy', 'value': validation.get('validation_accuracy')},
    {'metric': 'validation_macro_f1', 'value': validation.get('validation_macro_f1')},
    {'metric': 'gold_accuracy_ml', 'value': gold_metrics.get('gold_accuracy')},
    {'metric': 'gold_accuracy_heuristic', 'value': heuristic_gold_metrics.get('gold_accuracy')},
    {'metric': 'gold_holdout_skipped', 'value': corpus.get('gold_holdout_skipped_rows')},
]

print('Saved pipeline files:')
for name in ['preprocessing_pipeline.joblib', 'tfidf_vectorizer.joblib', 'logistic_classifier.joblib', 'training_report.json']:
    print(' -', settings.phase1_model_dir / name)

display(pd.DataFrame(summary_rows))
print('Gold eval (ML):')
display(pd.DataFrame(gold_metrics.get('gold_eval_rows', [])))


Saved pipeline files:
 - C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\artifacts\phase1_intent_model\preprocessing_pipeline.joblib
 - C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\artifacts\phase1_intent_model\tfidf_vectorizer.joblib
 - C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\artifacts\phase1_intent_model\logistic_classifier.joblib
 - C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\artifacts\phase1_intent_model\training_report.json


,metric,value
0,training_corpus_type,query_only
1,training_rows_total,231
2,train_rows_split,184
3,validation_rows_split,47
4,validation_accuracy,0.9787
5,validation_macro_f1,0.9749
6,gold_accuracy_ml,1.0
7,gold_accuracy_heuristic,1.0
8,gold_holdout_skipped,6


Gold eval (ML):


,query_id,expected_intent,predicted_intent,confidence,correct
0,q_incident_001,Incident_Report,Incident_Report,0.5562,True
1,q_procedure_001,Technical_Procedure,Technical_Procedure,0.6520,True
2,q_metadata_001,Metadata_Query,Metadata_Query,0.5490,True
3,q_factoid_001,Factoid,Factoid,0.6015,True
4,q_incident_002,Incident_Report,Incident_Report,0.4069,True
5,q_procedure_002,Technical_Procedure,Technical_Procedure,0.4753,True
6,q_metadata_002,Metadata_Query,Metadata_Query,0.4595,True
7,q_incident_003,Incident_Report,Incident_Report,0.4641,True


## Kiến trúc ML + luồng runtime

### Pipeline train (offline)

```text
seed + training_queries.jsonl + augmentation
  -> loại bỏ câu trùng gold-set (normalized exact match)
  -> preprocess: normalize + jargon + stem
  -> TfidfVectorizer(ngram_range=(1,2), max_features=60000)
  -> OneVsRestClassifier(LogisticRegression, class_weight=balanced)
  -> stratified val split 80/20
  -> evaluate gold-set (held-out)
  -> lưu joblib + training_report.json
```

### Pipeline infer (mỗi query)

```text
User query
  -> normalize_text + aviation jargon expansion
  -> [auto] ML predict + heuristic_intent()
  -> chọn intent + intent_source + confidence
  -> expand_query + rewrite_query
  -> ROUTING_POLICIES[intent] -> retrieval_plan
  -> InputAgentOutput -> Phase 2
```

### Thành phần sklearn (local, không API)

| Thành phần | Vai trò |
|------------|---------|
| `TfidfVectorizer` | Biến query thành vector TF-IDF (uni + bigram) |
| `LogisticRegression` | Phân lớp tuyến tính, xuất xác suất |
| `OneVsRestClassifier` | 4 intent = 4 binary classifier |


## Gold-set demo queries (8 câu)

Load từ `data/phase1_intent_gold_labels.jsonl` — **human label**, dùng cho:

- So sánh ML vs heuristic vs runtime `auto`
- Kiểm tra **routing strategy** có khớp `expected_strategy` không
- Export artifact demo cho Phase 2

| Cột | Ý nghĩa |
|-----|---------|
| `query_raw` | Câu hỏi user |
| `expected_intent` | Intent đúng (annotator) |
| `expected_strategy` | Strategy retrieval mong đợi sau routing |
| `annotator` / `notes` | Nguồn gán nhãn |

**8 câu này không nằm trong training corpus** (hold-out theo normalized text).


In [3]:
sample_queries = read_jsonl(settings.phase1_gold_labels_path)
pd.DataFrame(sample_queries)


,query_id,query_raw,expected_intent,expected_strategy,annotator,notes
0,q_incident_001,engine failure after takeoff with emergency re...,Incident_Report,semantic,project_team,narrative incident lookup
1,q_procedure_001,den bao ENG OIL PRESS sang thi lam gi?,Technical_Procedure,bm25,project_team,procedure / checklist style query
2,q_metadata_001,crosswind turbulence during final approach at ...,Metadata_Query,metadata_first,project_team,weather and runway metadata focus
3,q_factoid_001,what is the meaning of MEL in aviation?,Factoid,semantic,project_team,definition-style query
4,q_incident_002,bird strike on departure caused rejected takeoff,Incident_Report,semantic,project_team,wildlife strike narrative
5,q_procedure_002,engine warning checklist after takeoff,Technical_Procedure,bm25,project_team,checklist keyword routing
6,q_metadata_002,icing conditions reported during cruise at FL350,Metadata_Query,metadata_first,project_team,weather metadata filter
7,q_incident_003,smoke in cockpit during climb required immedia...,Incident_Report,semantic,project_team,smoke/fumes incident narrative


## So sánh: ML vs Heuristic vs Runtime (`auto`)

Cell dưới chạy **`build_output()`** với `input_intent_mode='auto'` (giống Streamlit mặc định).

| Cột | Ý nghĩa |
|-----|---------|
| `runtime_intent` | Intent cuối cùng Phase 1 trả về (ML hoặc heuristic) |
| `intent_source` | `"ml"` hoặc `"heuristic"` — mode thực tế đã chọn |
| `runtime_confidence` | Confidence của nguồn đã chọn |
| `ml_only_intent` | Chỉ chạy classifier, bỏ qua auto fallback |
| `heuristic_intent` | Rule baseline (`heuristic_intent()` trên query normalized) |
| `runtime_correct` | `runtime_intent == expected_intent` |
| `ml_only_correct` | ML thuần có đúng gold không |
| `heuristic_correct` | Heuristic có đúng gold không |
| `rewritten_query` | Query rewrite gửi kèm retrieval (theo intent) |

**Heuristic confidence** trong bảng tổng hợp = `0.60` cố định (baseline giả lập, không phải xác suất thật).


In [4]:
from aviation_rag.phase1_intent_training import preprocess_for_intent_ml

phase1 = Phase1HoangIntentRouting(settings)
phase1_rows = []
comparison_rows = []

for item in sample_queries:
    output = phase1.build_output(
        query_raw=item['query_raw'],
        query_id=item['query_id'],
        top_k=10,
    )
    phase1_rows.append(output)
    baseline_intent = heuristic_intent(output.query_normalized)
    ml_only_intent, ml_only_conf = phase1.intent_model.predict(
        preprocess_for_intent_ml(output.query_raw, use_stemming=settings.phase1_use_stemming)
    )
    comparison_rows.append({
        'query_id': output.query_id,
        'query_raw': output.query_raw,
        'expected_intent': item['expected_intent'],
        'runtime_intent': output.intent,
        'intent_source': output.intent_source,
        'runtime_confidence': round(float(output.intent_confidence), 4),
        'ml_only_intent': ml_only_intent,
        'ml_only_confidence': round(float(ml_only_conf), 4),
        'heuristic_intent': baseline_intent,
        'runtime_correct': output.intent == item['expected_intent'],
        'ml_only_correct': ml_only_intent == item['expected_intent'],
        'heuristic_correct': baseline_intent == item['expected_intent'],
        'rewritten_query': output.rewritten_query,
    })

comparison_frame = pd.DataFrame(comparison_rows)
summary_frame = pd.DataFrame([
    {
        'method': f"Runtime ({settings.input_intent_mode})",
        'intent_accuracy': comparison_frame['runtime_correct'].mean(),
        'confidence_mean': comparison_frame['runtime_confidence'].mean(),
        'confidence_source': 'ml or heuristic (auto)',
    },
    {
        'method': 'ML only',
        'intent_accuracy': comparison_frame['ml_only_correct'].mean(),
        'confidence_mean': comparison_frame['ml_only_confidence'].mean(),
        'confidence_source': 'model probability',
    },
    {
        'method': 'Heuristic baseline',
        'intent_accuracy': comparison_frame['heuristic_correct'].mean(),
        'confidence_mean': 0.60,
        'confidence_source': 'fixed baseline value',
    },
])

display(summary_frame)
comparison_frame


,method,intent_accuracy,confidence_mean,confidence_source
0,Runtime (auto),1.0,0.520563,ml or heuristic (auto)
1,ML only,1.0,0.520563,model probability
2,Heuristic baseline,1.0,0.600000,fixed baseline value


,query_id,query_raw,expected_intent,runtime_intent,intent_source,runtime_confidence,ml_only_intent,ml_only_confidence,heuristic_intent,runtime_correct,ml_only_correct,heuristic_correct,rewritten_query
0,q_incident_001,engine failure after takeoff with emergency re...,Incident_Report,Incident_Report,ml,0.5562,Incident_Report,0.5562,Incident_Report,True,True,True,aviation incident narrative lookup for: engine...
1,q_procedure_001,den bao ENG OIL PRESS sang thi lam gi?,Technical_Procedure,Technical_Procedure,ml,0.6520,Technical_Procedure,0.6520,Technical_Procedure,True,True,True,aviation troubleshooting and procedure lookup ...
2,q_metadata_001,crosswind turbulence during final approach at ...,Metadata_Query,Metadata_Query,ml,0.5490,Metadata_Query,0.5490,Metadata_Query,True,True,True,aviation metadata and operating condition look...
3,q_factoid_001,what is the meaning of MEL in aviation?,Factoid,Factoid,ml,0.6015,Factoid,0.6015,Factoid,True,True,True,direct aviation fact lookup for: what is the m...
4,q_incident_002,bird strike on departure caused rejected takeoff,Incident_Report,Incident_Report,ml,0.4069,Incident_Report,0.4069,Incident_Report,True,True,True,aviation incident narrative lookup for: bird s...
5,q_procedure_002,engine warning checklist after takeoff,Technical_Procedure,Technical_Procedure,ml,0.4753,Technical_Procedure,0.4753,Technical_Procedure,True,True,True,aviation troubleshooting and procedure lookup ...
6,q_metadata_002,icing conditions reported during cruise at FL350,Metadata_Query,Metadata_Query,ml,0.4595,Metadata_Query,0.4595,Metadata_Query,True,True,True,aviation metadata and operating condition look...
7,q_incident_003,smoke in cockpit during climb required immedia...,Incident_Report,Incident_Report,ml,0.4641,Incident_Report,0.4641,Incident_Report,True,True,True,aviation incident narrative lookup for: smoke ...


## Dynamic Retrieval Routing

Sau khi classifier dự đoán intent, Phase 1 chọn retrieval strategy cho Phase 2:

- `Incident_Report` → semantic
- `Technical_Procedure` → bm25
- `Metadata_Query` → metadata_first
- `Factoid` → semantic (fallback hybrid)


In [5]:
routing_frame = pd.DataFrame([
    {
        'query_id': row.query_id,
        'intent': row.intent,
        'strategy': row.retrieval_plan.strategy,
        'fallback_strategy': row.retrieval_plan.fallback_strategy,
        'filters': row.retrieval_plan.filters,
        'routing_reason': row.retrieval_plan.routing_reason,
    }
    for row in phase1_rows
])
routing_frame


,query_id,intent,strategy,fallback_strategy,filters,routing_reason
0,q_incident_001,Incident_Report,semantic,hybrid,{},Narrative incident queries benefit from semant...
1,q_procedure_001,Technical_Procedure,bm25,hybrid,{'document_type': 'procedure'},Procedure-style queries favor keyword-heavy ch...
2,q_metadata_001,Metadata_Query,metadata_first,bm25,{'prefer_metadata': True},Metadata queries should prioritize structured ...
3,q_factoid_001,Factoid,semantic,hybrid,{'answer_style': 'short'},Factoid queries need concise semantic lookup w...
4,q_incident_002,Incident_Report,semantic,hybrid,{},Narrative incident queries benefit from semant...
5,q_procedure_002,Technical_Procedure,bm25,hybrid,{'document_type': 'procedure'},Procedure-style queries favor keyword-heavy ch...
6,q_metadata_002,Metadata_Query,metadata_first,bm25,{'prefer_metadata': True},Metadata queries should prioritize structured ...
7,q_incident_003,Incident_Report,semantic,hybrid,{},Narrative incident queries benefit from semant...


## So sánh BM25 vs Semantic (Phase 2, query procedure)

**Mục đích:** minh họa vì sao `Technical_Procedure` route sang **bm25** — keyword checklist/procedure khớp lexical tốt hơn dense vector đôi khi.

Cùng một query procedure, chạy Phase 2 hai lần:

| Strategy | Công thức `final_score` (trong `phase2_retrieval_engine.py`) |
|----------|----------------------------------------------------------------|
| **bm25** | `0.85 × bm25 + 0.15 × metadata` |
| **semantic** | `0.85 × semantic + 0.15 × metadata` |

**Cột quan trọng**

- `top_doc_changed` — top-1 có đổi doc không giữa hai strategy
- `bm25_score` / `semantic_score` — thành phần dominant của từng strategy
- `final_score` — điểm xếp hạng cuối

Đây là **retrieval experiment**, không phải metric train Phase 1.


In [6]:
retrieval_settings = replace(
    settings,
    phase2_embedding_model='tfidf_svd_fallback',
    phase2_index_dir=settings.artifacts_dir / 'phase2_index_fast',
    retrieval_max_docs=min(settings.retrieval_max_docs, 6000),
    retrieval_svd_components=min(settings.retrieval_svd_components, 96),
)
ensure_artifact_dirs(retrieval_settings)

bm25_probe_query = 'qrh checklist for engine fire warning'
probe_phase1 = Phase1HoangIntentRouting(retrieval_settings)
retrieval = Phase2SanFaissRetrieval(retrieval_settings)

strategy_outputs = []
detail_rows = []
try:
    for strategy in ['bm25', 'semantic']:
        input_row = probe_phase1.build_output(
            bm25_probe_query, query_id=f'probe_{strategy}', top_k=5, strategy=strategy
        )
        output = retrieval.retrieve(input_row)
        top_docs = output.topk_docs[:3]
        top_doc = top_docs[0] if top_docs else None
        scores = top_doc.scores if top_doc else {}
        strategy_outputs.append({
            'strategy': strategy,
            'uses_bm25_in_final': strategy == 'bm25',
            'fusion_method': output.retrieval_diagnostics.get('fusion_method'),
            'score_weights': str(output.retrieval_diagnostics.get('score_weights', {})),
            'top_doc_id': top_doc.doc_id if top_doc else None,
            'top_doc_type': top_doc.metadata.get('document_type') if top_doc else None,
            'semantic_score': round(float(scores.get('semantic', 0.0)), 4),
            'bm25_score': round(float(scores.get('bm25', 0.0)), 4),
            'metadata_score': round(float(scores.get('metadata', 0.0)), 4),
            'final_score': round(float(scores.get('final', 0.0)), 4),
        })
        for rank, doc in enumerate(top_docs, start=1):
            detail_rows.append({
                'strategy': strategy,
                'rank': rank,
                'doc_id': doc.doc_id,
                'doc_type': doc.metadata.get('document_type'),
                'semantic': round(float(doc.scores.get('semantic', 0.0)), 4),
                'bm25': round(float(doc.scores.get('bm25', 0.0)), 4),
                'final': round(float(doc.scores.get('final', 0.0)), 4),
            })

    bm25_comparison = pd.DataFrame(strategy_outputs)
    bm25_top3 = pd.DataFrame(detail_rows)

    bm25_top = bm25_comparison.loc[bm25_comparison['strategy'] == 'bm25', 'top_doc_id'].iloc[0]
    sem_top = bm25_comparison.loc[bm25_comparison['strategy'] == 'semantic', 'top_doc_id'].iloc[0]
    bm25_row = bm25_comparison[bm25_comparison['strategy'] == 'bm25'].iloc[0]
    sem_row = bm25_comparison[bm25_comparison['strategy'] == 'semantic'].iloc[0]

    diff_summary = pd.DataFrame([{
        'query': bm25_probe_query,
        'top_doc_changed': bm25_top != sem_top,
        'bm25_top_doc': bm25_top,
        'semantic_top_doc': sem_top,
        'bm25_final_score': bm25_row['final_score'],
        'semantic_final_score': sem_row['final_score'],
        'bm25_weight_dominant_score': bm25_row['bm25_score'],
        'semantic_weight_dominant_score': sem_row['semantic_score'],
    }])

    print('=== Tổng hợp BM25 vs Semantic ===')
    display(diff_summary)
    print('=== Top-1 theo strategy ===')
    display(bm25_comparison)
    print('=== Top-3 chi tiết ===')
    bm25_top3
except Exception as exc:
    pd.DataFrame([{'error': str(exc)}])


=== Tổng hợp BM25 vs Semantic ===


,query,top_doc_changed,bm25_top_doc,semantic_top_doc,bm25_final_score,semantic_final_score,bm25_weight_dominant_score,semantic_weight_dominant_score
0,qrh checklist for engine fire warning,True,687140,653370,0.8211,0.93,0.7896,1.0


=== Top-1 theo strategy ===


,strategy,uses_bm25_in_final,fusion_method,score_weights,top_doc_id,top_doc_type,semantic_score,bm25_score,metadata_score,final_score
0,bm25,True,weighted_linear_fusion,"{'bm25': 0.85, 'metadata': 0.15}",687140,procedure,0.6532,0.7896,0.75,0.8211
1,semantic,False,weighted_linear_fusion,"{'semantic': 0.85, 'metadata': 0.15}",653370,procedure,1.0000,0.3153,0.40,0.9300


=== Top-3 chi tiết ===


## Export Contract cho Phase 2 (San)

Mỗi dòng JSONL = schema **`InputAgentOutput`** (`aviation_rag/schemas.py`):

| Field | Bắt buộc | Ý nghĩa |
|-------|----------|---------|
| `query_id` | ✓ | ID duy nhất |
| `query_raw` | ✓ | Query gốc |
| `query_normalized` | ✓ | Sau normalize |
| `intent` | ✓ | Intent đã route |
| `intent_confidence` | ✓ | Confidence ML hoặc heuristic |
| `intent_source` | ✓ | `"ml"` / `"heuristic"` |
| `expanded_queries` | ✓ | Biến thể mở rộng query |
| `rewritten_query` | ✓ | Prefix theo intent cho retrieval |
| `retrieval_plan` | ✓ | strategy, fallback, filters, top_k |
| `created_at` | ✓ | Timestamp UTC |

File output: `artifacts/phase1_hoang_intent_routing_output.jsonl` — Phase 2 đọc qua contract adapter hoặc graph trực tiếp.


In [7]:
write_jsonl(settings.phase1_output_path, phase1_rows)
required = [
    'query_id', 'query_raw', 'query_normalized', 'intent', 'intent_confidence',
    'intent_source', 'expanded_queries', 'rewritten_query', 'retrieval_plan', 'created_at'
]
for row in phase1_rows:
    payload = row.model_dump(mode='json')
    missing = [key for key in required if key not in payload]
    if missing:
        raise ValueError(f'Missing keys in {payload.get("query_id")}: {missing}')
print(f'Wrote {len(phase1_rows)} rows to {settings.phase1_output_path}')
print('Phase 1 contract validation passed.')


Wrote 8 rows to C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\artifacts\phase1_hoang_intent_routing_output.jsonl
Phase 1 contract validation passed.


## Đánh giá Phase 1

Metrics:

- **Intent accuracy**: số query phân loại đúng intent
- **Routing accuracy**: số query chọn đúng retrieval strategy
- **Confidence**: min/mean/max từ Logistic Regression


In [8]:
expected = {item['query_id']: item for item in sample_queries}

evaluation_rows = []
for row in phase1_rows:
    gold = expected.get(row.query_id, {})
    evaluation_rows.append({
        'query_id': row.query_id,
        'query_raw': row.query_raw,
        'predicted_intent': row.intent,
        'expected_intent': gold.get('expected_intent'),
        'intent_correct': row.intent == gold.get('expected_intent'),
        'predicted_strategy': row.retrieval_plan.strategy,
        'expected_strategy': gold.get('expected_strategy'),
        'strategy_correct': row.retrieval_plan.strategy == gold.get('expected_strategy'),
        'confidence': round(float(row.intent_confidence), 4),
    })

evaluation_frame = pd.DataFrame(evaluation_rows)
summary = {
    'sample_size': len(evaluation_rows),
    'intent_accuracy': round(evaluation_frame['intent_correct'].mean(), 4),
    'routing_accuracy': round(evaluation_frame['strategy_correct'].mean(), 4),
    'confidence_min': round(evaluation_frame['confidence'].min(), 4),
    'confidence_mean': round(evaluation_frame['confidence'].mean(), 4),
    'confidence_max': round(evaluation_frame['confidence'].max(), 4),
}
print(summary)
evaluation_frame


{'sample_size': 8, 'intent_accuracy': np.float64(1.0), 'routing_accuracy': np.float64(1.0), 'confidence_min': np.float64(0.4069), 'confidence_mean': np.float64(0.5206), 'confidence_max': np.float64(0.652)}


,query_id,query_raw,predicted_intent,expected_intent,intent_correct,predicted_strategy,expected_strategy,strategy_correct,confidence
0,q_incident_001,engine failure after takeoff with emergency re...,Incident_Report,Incident_Report,True,semantic,semantic,True,0.5562
1,q_procedure_001,den bao ENG OIL PRESS sang thi lam gi?,Technical_Procedure,Technical_Procedure,True,bm25,bm25,True,0.6520
2,q_metadata_001,crosswind turbulence during final approach at ...,Metadata_Query,Metadata_Query,True,metadata_first,metadata_first,True,0.5490
3,q_factoid_001,what is the meaning of MEL in aviation?,Factoid,Factoid,True,semantic,semantic,True,0.6015
4,q_incident_002,bird strike on departure caused rejected takeoff,Incident_Report,Incident_Report,True,semantic,semantic,True,0.4069
5,q_procedure_002,engine warning checklist after takeoff,Technical_Procedure,Technical_Procedure,True,bm25,bm25,True,0.4753
6,q_metadata_002,icing conditions reported during cruise at FL350,Metadata_Query,Metadata_Query,True,metadata_first,metadata_first,True,0.4595
7,q_incident_003,smoke in cockpit during climb required immedia...,Incident_Report,Incident_Report,True,semantic,semantic,True,0.4641
